# EPI Recorder - NEXUS Ventures Demo

This version keeps the old demo's fast flow, but shows what changed in **EPI v2.8.2**.

**5 cells. ~90 seconds. Run All.**

What is new compared to the earlier demo:
- the run uses **`epi_policy.json`**
- EPI produces **`analysis.json`** before sealing
- the demo shows a **policy-grounded fault** instead of only raw evidence
- it appends a **human review** as `review.json`
- it still proves **tamper detection** and opens the embedded viewer


In [ ]:
# @title 1. Install EPI Recorder { display-mode: "form" }
INSTALL_SOURCE = "pypi"  # @param ["pypi", "github"]
GITHUB_REF = "main"

if INSTALL_SOURCE == "pypi":
    %pip install -q --force-reinstall --no-cache-dir epi-recorder==2.8.2
else:
    repo_url = f"git+https://github.com/mohdibrahimaiml/epi-recorder.git@{GITHUB_REF}"
    %pip install -q --force-reinstall --no-cache-dir {repo_url}

import epi_core
print(f"[OK] Installed epi-recorder {epi_core.__version__}")
print("This demo now focuses on policy + fault analysis, not just recording.")
print("Defaulting to PyPI now installs the released 2.8.2 viewer and front-door fixes.")


In [ ]:
# @title 2. Run AI Agent with EPI Policy + Fault Analysis { display-mode: "form" }
import json
import shutil
from pathlib import Path

workspace = Path("/content/nexus_ventures_demo")
workspace.mkdir(parents=True, exist_ok=True)

for stale in [
    workspace / 'nexus_demo.epi',
    workspace / 'nexus_demo_reviewed.epi',
    workspace / 'NEXUS_FORGED.epi',
    workspace / 'nexus_demo_workflow.py',
    workspace / 'epi_policy.json',
]:
    if stale.exists():
        if stale.is_file():
            stale.unlink()
        else:
            shutil.rmtree(stale)

policy = {
    "system_name": "loan-underwriting-agent",
    "system_version": "2.8.2-demo",
    "policy_version": "2026-03-17",
    "profile_id": "finance.loan-underwriting",
    "rules": [
        {
            "id": "R001",
            "name": "Do Not Exceed Balance",
            "severity": "critical",
            "description": "The agent must not approve an amount above the known balance.",
            "type": "constraint_guard",
            "watch_for": ["balance", "available_balance", "limit"],
            "violation_if": "approved_amount > balance"
        },
        {
            "id": "R002",
            "name": "Verify Identity Before Refund",
            "severity": "high",
            "description": "Identity verification must happen before refund.",
            "type": "sequence_guard",
            "required_before": "refund",
            "must_call": "verify_identity"
        },
        {
            "id": "R003",
            "name": "Human Approval Above 10000",
            "severity": "high",
            "description": "Amounts above 10,000 require human approval.",
            "type": "threshold_guard",
            "watch_for": ["amount", "approved_amount", "loan_amount"],
            "threshold_value": 10000,
            "required_action": "human_approval"
        },
        {
            "id": "R004",
            "name": "Never Output Secrets",
            "severity": "critical",
            "description": "The system must not expose secret-like tokens in output.",
            "type": "prohibition_guard",
            "pattern": "sk-[A-Za-z0-9]+"
        }
    ]
}

(workspace / "epi_policy.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")

agent_code = r'''
import json
from pathlib import Path
from epi_recorder import record

output = Path("/content/nexus_ventures_demo/nexus_demo.epi")

with record(
    str(output),
    workflow_name="Nexus Ventures Financial Fault Demo",
    goal="Show that EPI records AI decisions and pinpoints mistakes",
    notes="Intentional policy and heuristic violations for investor demonstration",
    metadata_tags=["nexus-ventures", "investor-demo", "fault-analysis", "finance"],
) as epi:
    epi.log_step("tool.response", {
        "tool": "fetch_account_context",
        "account_id": "ACC-4401",
        "customer_id": "CUST-9001",
        "balance": 500.0,
        "currency": "USD",
    })

    epi.log_step("llm.error", {
        "error": "Rate limit exceeded while fetching sanctions data",
        "provider": "demo-llm",
    })

    epi.log_step("tool.call", {
        "tool": "approve_loan_disbursement",
        "amount": 2000.0,
        "account_id": "ACC-4401",
        "reason": "urgent disbursement",
    })

    epi.log_step("tool.call", {
        "tool": "process_refund",
        "amount": 200.0,
        "reference": "REF-2026-001",
    })

    epi.log_step("tool.call", {
        "tool": "approve_wire_transfer",
        "amount": 15000.0,
        "destination": "vendor-settlement",
    })

    epi.log_step("llm.response", {
        "text": "Escalation note: temporary key observed as sk-ABC123SECRET for retry flow.",
        "model": "demo-llm",
    })

    epi.log_step("tool.response", {
        "tool": "create_case",
        "case_reference": "CASE-7702",
        "status": "pending_manual_review",
    })

print(f"Created demo artifact: {output}")
'''

(workspace / "nexus_demo_workflow.py").write_text(agent_code, encoding="utf-8")

%cd /content/nexus_ventures_demo
!python nexus_demo_workflow.py

epi_file = workspace / "nexus_demo.epi"
assert epi_file.exists(), 'Expected nexus_demo.epi to be created'
print(f"\n[PACKAGE] Evidence artifact: {epi_file.name} ({epi_file.stat().st_size / 1024:.0f} KB)")
print("What changed from the old demo: this artifact now includes policy.json and analysis.json.")


In [ ]:
# @title 3. Verify + Show Fault Analysis + Append Human Review { display-mode: "form" }
import json, zipfile, shutil, sys
from pathlib import Path
from epi_cli.keys import KeyManager, generate_default_keypair_if_missing
from epi_core.review import ReviewRecord, add_review_to_artifact, make_review_entry

epi_file = Path('/content/nexus_ventures_demo/nexus_demo.epi')
assert epi_file.exists(), 'Missing nexus_demo.epi - run cell 2 first'

print('=' * 70)
print('VERIFY ARTIFACT')
!{sys.executable} -m epi_cli.main verify {epi_file}

print('\n' + '=' * 70)
print('FAULT ANALYZER OUTPUT')
!{sys.executable} -m epi_cli.main analyze {epi_file}

with zipfile.ZipFile(epi_file, 'r') as zf:
    policy_json = json.loads(zf.read('policy.json').decode('utf-8'))
    analysis_json = json.loads(zf.read('analysis.json').decode('utf-8'))

print('\nEmbedded files now present:')
print('- policy.json rules =', len(policy_json.get('rules', [])))
print('- primary fault =', analysis_json['primary_fault'].get('fault_type'))
print('- violated rule =', analysis_json['primary_fault'].get('rule_id'))
print('- why it matters =', analysis_json['primary_fault'].get('why_it_matters'))

reviewed = Path('/content/nexus_ventures_demo/nexus_demo_reviewed.epi')
if reviewed.exists():
    reviewed.unlink()
shutil.copy2(epi_file, reviewed)
generate_default_keypair_if_missing(console_output=False)
review = ReviewRecord(
    reviewed_by='nexus.demo.reviewer@epilabs.org',
    reviews=[
        make_review_entry(
            fault=analysis_json['primary_fault'],
            outcome='confirmed_fault',
            notes='Confirmed for demo: this violation was intentionally introduced to show EPI review.',
            reviewer='nexus.demo.reviewer@epilabs.org',
        )
    ],
)
review.sign(KeyManager().load_private_key('default'))
add_review_to_artifact(reviewed, review)
assert reviewed.exists(), 'Expected reviewed artifact to be created'
print('\n[REVIEW] Added review.json to nexus_demo_reviewed.epi')
print('What changed from the old demo: human review is now preserved inside the artifact.')


In [ ]:
# @title 4. Tamper Test - Try to Forge the Evidence { display-mode: "form" }
import json, zipfile
from pathlib import Path

import sys
source_epi = Path('/content/nexus_ventures_demo/nexus_demo_reviewed.epi')
tampered = Path('/content/nexus_ventures_demo/NEXUS_FORGED.epi')
assert source_epi.exists(), 'Missing nexus_demo_reviewed.epi - run cell 3 first'
if tampered.exists():
    tampered.unlink()

with zipfile.ZipFile(source_epi, 'r') as src, zipfile.ZipFile(tampered, 'w', compression=zipfile.ZIP_DEFLATED) as dst:
    for item in src.infolist():
        data = src.read(item.filename)
        if item.filename == 'analysis.json':
            payload = json.loads(data.decode('utf-8'))
            payload['fault_detected'] = False
            payload.setdefault('summary', {})
            payload['summary']['headline'] = 'FORGED: analysis changed after sealing'
            data = json.dumps(payload, indent=2).encode('utf-8')
        dst.writestr(item, data)

print('=' * 60)
print('REVIEWED ORIGINAL:')
!{sys.executable} -m epi_cli.main verify {source_epi}
print('\n' + '=' * 60)
print('FORGED:')
!{sys.executable} -m epi_cli.main verify {tampered}
print('=' * 60)
assert tampered.exists(), 'Expected forged artifact to be created'
print('What changed from the old demo: this now shows tampering against a policy-analyzed artifact, not just a raw trace.')


In [ ]:
# @title 5. Open the EPI Viewer { display-mode: "form" }
VIEW_TARGET = 'tampered'  # @param ["reviewed", "tampered", "generated"]

import html, json, re, tempfile, zipfile
from pathlib import Path
from IPython.display import HTML, display
from epi_cli.view import _build_viewer_context
from epi_core.container import EPIContainer
from epi_core.schemas import ManifestModel

generated_epi = Path('/content/nexus_ventures_demo/nexus_demo.epi')
reviewed_epi = Path('/content/nexus_ventures_demo/nexus_demo_reviewed.epi')
tampered_epi = Path('/content/nexus_ventures_demo/NEXUS_FORGED.epi')
targets = {
    'generated': generated_epi,
    'reviewed': reviewed_epi,
    'tampered': tampered_epi,
}
epi_file = targets[VIEW_TARGET]
assert generated_epi.exists(), 'Missing generated .epi file - run cell 2 first'
assert epi_file.exists(), f'Missing {epi_file.name} - run the earlier cells first'

with zipfile.ZipFile(epi_file) as z:
    names = set(z.namelist())
    manifest = json.loads(z.read('manifest.json'))
    steps_raw = z.read('steps.jsonl').decode().strip()
    steps = [json.loads(line) for line in steps_raw.split('\n') if line]
    analysis = json.loads(z.read('analysis.json').decode()) if 'analysis.json' in names else None
    policy = json.loads(z.read('policy.json').decode()) if 'policy.json' in names else None
    review = json.loads(z.read('review.json').decode()) if 'review.json' in names else None

with tempfile.TemporaryDirectory() as tmpdir:
    source_dir = Path(tmpdir)
    (source_dir / 'steps.jsonl').write_text(steps_raw + ('\n' if steps_raw and not steps_raw.endswith('\n') else ''), encoding='utf-8')
    if 'environment.json' in names:
        (source_dir / 'environment.json').write_bytes(zipfile.ZipFile(epi_file).read('environment.json'))
    if analysis is not None:
        (source_dir / 'analysis.json').write_text(json.dumps(analysis, indent=2), encoding='utf-8')
    if policy is not None:
        (source_dir / 'policy.json').write_text(json.dumps(policy, indent=2), encoding='utf-8')
    if review is not None:
        (source_dir / 'review.json').write_text(json.dumps(review, indent=2), encoding='utf-8')
    viewer = EPIContainer._create_embedded_viewer(source_dir, ManifestModel.model_validate(manifest))

view_context = json.dumps(_build_viewer_context(epi_file)).replace('</', '<\\/')
if 'id="epi-view-context"' in viewer:
    viewer = re.sub(
        r'<script id="epi-view-context" type="application/json">.*?</script>',
        f'<script id="epi-view-context" type="application/json">{view_context}</script>',
        viewer,
        flags=re.DOTALL,
    )
elif '</head>' in viewer:
    viewer = viewer.replace(
        '</head>',
        f'<script id="epi-view-context" type="application/json">{view_context}</script></head>'
    )
else:
    viewer = f'<script id="epi-view-context" type="application/json">{view_context}</script>' + viewer

escaped = html.escape(viewer)
sig = manifest.get('signature', '')[:30]
using_new_viewer = 'Evidence Packaged Infrastructure for AI' in viewer
primary_fault = (analysis or {}).get('primary_fault') or {}
reviewed_by = (review or {}).get('reviewed_by') or (((review or {}).get('reviews') or [{}])[0].get('reviewer') if review else None)
trust = _build_viewer_context(epi_file)
trust_label = 'Tampered' if not trust.get('integrity_ok') else ('Signed' if trust.get('signature_valid') else 'Unsigned')
banner = '#dc2626' if trust_label == 'Tampered' else '#10b981'

display(HTML(f'''
<div style="border:2px solid {banner}; border-radius:8px; overflow:hidden; margin:10px 0;">
  <div style="background:{banner}; color:white; padding:10px 16px; font-weight:bold;">
    EPI Viewer ({VIEW_TARGET}) - {epi_file.name} - Trust: {trust_label} - Sig: {sig}...
  </div>
  <iframe srcdoc="{escaped}" width="100%" height="680" style="border:none;" sandbox="allow-scripts allow-same-origin"></iframe>
</div>
'''))

print('Steps captured:', len(steps))
print('Viewer skin:', 'v2.8.0 current viewer' if using_new_viewer else 'older embedded viewer')
print('Primary fault shown in viewer:', primary_fault.get('fault_type', 'Unavailable'))
print('Reviewed by:', reviewed_by or 'Unavailable')
print('Trust state:', trust_label)
print('\nSet VIEW_TARGET = "reviewed" to switch back to the signed reviewed artifact, or keep "tampered" to demo the forged file.')

try:
    from google.colab import files
    files.download(str(generated_epi))
    # files.download(str(epi_file))  # uncomment if you also want the currently viewed artifact
except Exception:
    pass
